In [1]:
import torch
import torch.nn as nn
import torchvision.transforms as transforms
import torchvision.datasets as dsets
from torch import Tensor
import torch.nn.functional as F
from torch.nn import init, Parameter
import pdb
import math
import numpy as np
import torch.nn.functional as F
from torch.autograd import Variable
import os 
import random
import pandas as pd
from copy import deepcopy
from typing import Dict
import transformers
from torch import Tensor
from torch.nn import init, Parameter
import torch.nn.functional as F
import pdb
import math
from collections import defaultdict
import numpy as np
import os 
import json
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, recall_score, f1_score, average_precision_score, precision_score
from torch.utils.data import DataLoader,Dataset,TensorDataset
import torch.optim as optim

In [2]:
hours_data = 24
data_path = "physionet.org/files/challenge-2012/1.0.0/phase1/all_subjects_data_48_hours/"
subject= np.load(os.path.join(data_path,"subjects_data.npz"),allow_pickle=True)
S = np.squeeze(subject["statics_data"], axis=1)
targets = subject["targets_data"]
timeseries = np.array(subject["timeseries_imp_data"], dtype=float)[:, :hours_data, :]
timeseries_org = np.array(subject["timeseries_data"], dtype=float)[:, :hours_data, :]
timedelta = np.array(subject["delta_time_data"], dtype=float)[:, :hours_data, :]
icu_mor = targets

In [3]:
unique, counts = np.unique(icu_mor, return_counts=True)

dict(zip(unique, counts))

{np.int64(0): np.int64(10281), np.int64(1): np.int64(1707)}

In [4]:
repeats =timeseries.shape[1]
temporal_statics = np.tile(S, repeats).reshape(S.shape[0], repeats,S.shape[-1])
timedelta_statics= np.zeros((S.shape[0], repeats,S.shape[-1]))
temporal_statics.shape, timedelta_statics.shape

((11988, 24, 2), (11988, 24, 2))

In [5]:
temporal_data = np.concatenate((timeseries_org, temporal_statics), axis=-1)
temporal_timedelta = np.concatenate((timedelta, np.zeros_like(temporal_statics)), axis=-1)
temporal_timedelta[np.isnan(temporal_timedelta)] = 999
temporal_timedelta[np.isinf(temporal_timedelta)] = 999
temporal_data_features = np.concatenate((timeseries_org, temporal_statics), axis=-1)

In [6]:
temporal_data_features

array([[[34.  ,   nan, 12.  , ...,  7.32, 17.  ,  7.  ],
        [34.  ,   nan, 11.  , ...,   nan, 17.  ,  7.  ],
        [34.  ,   nan,   nan, ...,   nan, 17.  ,  7.  ],
        ...,
        [34.  ,   nan,   nan, ...,   nan, 17.  ,  7.  ],
        [34.  ,   nan,   nan, ...,   nan, 17.  ,  7.  ],
        [34.  ,   nan,   nan, ...,   nan, 17.  ,  7.  ]],

       [[42.  ,   nan,   nan, ...,   nan,  4.  , 10.  ],
        [42.  ,   nan,   nan, ...,   nan,  4.  , 10.  ],
        [42.  ,  3.1 , 41.  , ...,   nan,  4.  , 10.  ],
        ...,
        [42.  ,   nan,   nan, ...,   nan,  4.  , 10.  ],
        [42.  ,   nan,   nan, ...,   nan,  4.  , 10.  ],
        [42.  ,   nan,   nan, ...,   nan,  4.  , 10.  ]],

       [[79.  ,   nan,   nan, ...,  7.43, 18.  ,  8.  ],
        [79.  ,   nan,   nan, ...,  7.32, 18.  ,  8.  ],
        [79.  ,   nan,   nan, ...,  7.39, 18.  ,  8.  ],
        ...,
        [79.  ,   nan,   nan, ...,   nan, 18.  ,  8.  ],
        [79.  ,   nan,   nan, ...,   nan, 18.

In [7]:
timeseries.shape,timeseries_org.shape, temporal_data_features.shape

((11988, 24, 26), (11988, 24, 26), (11988, 24, 28))

In [8]:
temporal_data

array([[[34.  ,   nan, 12.  , ...,  7.32, 17.  ,  7.  ],
        [34.  ,   nan, 11.  , ...,   nan, 17.  ,  7.  ],
        [34.  ,   nan,   nan, ...,   nan, 17.  ,  7.  ],
        ...,
        [34.  ,   nan,   nan, ...,   nan, 17.  ,  7.  ],
        [34.  ,   nan,   nan, ...,   nan, 17.  ,  7.  ],
        [34.  ,   nan,   nan, ...,   nan, 17.  ,  7.  ]],

       [[42.  ,   nan,   nan, ...,   nan,  4.  , 10.  ],
        [42.  ,   nan,   nan, ...,   nan,  4.  , 10.  ],
        [42.  ,  3.1 , 41.  , ...,   nan,  4.  , 10.  ],
        ...,
        [42.  ,   nan,   nan, ...,   nan,  4.  , 10.  ],
        [42.  ,   nan,   nan, ...,   nan,  4.  , 10.  ],
        [42.  ,   nan,   nan, ...,   nan,  4.  , 10.  ]],

       [[79.  ,   nan,   nan, ...,  7.43, 18.  ,  8.  ],
        [79.  ,   nan,   nan, ...,  7.32, 18.  ,  8.  ],
        [79.  ,   nan,   nan, ...,  7.39, 18.  ,  8.  ],
        ...,
        [79.  ,   nan,   nan, ...,   nan, 18.  ,  8.  ],
        [79.  ,   nan,   nan, ...,   nan, 18.

In [13]:
freq_list , timeseries_last_obs_data= [], []
nb_pats, seq, n_features = temporal_data.shape
for i in range(nb_pats):
    data_patient=  np.expand_dims(temporal_data[i,:,:], axis=0)
    nan_counts = np.sum(np.isnan(data_patient), axis=(0, 1))
    freq_list.append(np.repeat(np.expand_dims(nan_counts, axis=0), repeats, axis=0))
    # Last observation record
    Index_Last=(~np.isnan(temporal_data[i,:,:])).cumsum(0).argmax(0)
    Last_Indices = np.reshape(Index_Last,(1,n_features))
    Last_Values = np.take_along_axis(temporal_data[i,:,:], Last_Indices, axis = 0)
    timeseries_last_obs_data.append(np.repeat(Last_Values, repeats, axis=0))
freqs = np.stack(freq_list)
last_obs_data=np.stack(timeseries_last_obs_data)

In [11]:
freqs=1

In [14]:
freqs

array([[[ 0, 24, 21, ..., 17,  0,  0],
        [ 0, 24, 21, ..., 17,  0,  0],
        [ 0, 24, 21, ..., 17,  0,  0],
        ...,
        [ 0, 24, 21, ..., 17,  0,  0],
        [ 0, 24, 21, ..., 17,  0,  0],
        [ 0, 24, 21, ..., 17,  0,  0]],

       [[ 0, 23, 23, ..., 24,  0,  0],
        [ 0, 23, 23, ..., 24,  0,  0],
        [ 0, 23, 23, ..., 24,  0,  0],
        ...,
        [ 0, 23, 23, ..., 24,  0,  0],
        [ 0, 23, 23, ..., 24,  0,  0],
        [ 0, 23, 23, ..., 24,  0,  0]],

       [[ 0, 24, 22, ..., 17,  0,  0],
        [ 0, 24, 22, ..., 17,  0,  0],
        [ 0, 24, 22, ..., 17,  0,  0],
        ...,
        [ 0, 24, 22, ..., 17,  0,  0],
        [ 0, 24, 22, ..., 17,  0,  0],
        [ 0, 24, 22, ..., 17,  0,  0]],

       ...,

       [[ 0, 23, 23, ..., 24,  0,  0],
        [ 0, 23, 23, ..., 24,  0,  0],
        [ 0, 23, 23, ..., 24,  0,  0],
        ...,
        [ 0, 23, 23, ..., 24,  0,  0],
        [ 0, 23, 23, ..., 24,  0,  0],
        [ 0, 23, 23, ..., 24,  0

In [19]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader,Dataset,TensorDataset
from sklearn.model_selection import KFold, train_test_split

def dataloaders_double_cv_cohort(train_x, train_t,  train_last,  train_freq, train_y, 
                          test_x, test_t,  test_last,  test_freq,  test_y, BATCH_SIZE=64):
        
    def collate_fn(batch):
        return tuple(zip(*batch))

    train_dataset = TensorDataset(torch.tensor(train_x,dtype=torch.float32),
                                  torch.tensor(train_t,dtype=torch.float32),
                                  torch.tensor(train_last,dtype=torch.float32),
                                  torch.tensor(train_freq,dtype=torch.float32),
                                  torch.tensor(train_y,dtype=torch.float32))  
    
    
    valid_dataset = TensorDataset(torch.tensor(test_x,dtype=torch.float32),
                                  torch.tensor(test_t,dtype=torch.float32),
                                  torch.tensor(test_last,dtype=torch.float32),
                                  torch.tensor(test_freq,dtype=torch.float32),
                                  torch.tensor(test_y,dtype=torch.float32))  
    
    
    
    train_data_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=4,
    )

    valid_data_loader = DataLoader(
        valid_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=4,
    )
    
    return train_data_loader, valid_data_loader

def cv_fold_splits_cohorts(data, time_data, last_data,
                           features_freqs, target, n_fold=10):
    from sklearn.model_selection import StratifiedKFold, StratifiedShuffleSplit

    test_data_loader, training_data, validation_data = [], [], []

    kfold = StratifiedKFold(n_splits=n_fold, shuffle=True, random_state=n_fold)
    strat_target = target.squeeze() if target.ndim > 1 else target

    # ── StratifiedShuffleSplit for the inner validation split ─────────
    sss = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=n_fold)

    for index, (train_index, test_index) in enumerate(
            kfold.split(data, strat_target)):
        print(f'<------- OUTER FOLD {index + 1} ------->')

        # ── Raw outer split (un-normalized) ──────────────────────────
        x_train,      x_test      = data[train_index],          data[test_index]
        x_train_last, x_test_last = last_data[train_index],     last_data[test_index]
        x_train_freq, x_test_freq = features_freqs[train_index],features_freqs[test_index]
        x_train_t,    x_test_t    = time_data[train_index],     time_data[test_index]
        y_train,      y_test      = target[train_index],        target[test_index]

        # ── Inner split via StratifiedShuffleSplit (mirrors cv_fold_splits) ──
        sss.get_n_splits(x_train, y_train)
        tr_idx, val_idx = next(sss.split(x_train, y_train))   # ← changed

        x_tr_in,    x_val_in    = x_train[tr_idx],       x_train[val_idx]
        t_tr_in,    t_val_in    = x_train_t[tr_idx],     x_train_t[val_idx]
        last_tr_in, last_val_in = x_train_last[tr_idx],  x_train_last[val_idx]
        freq_tr_in, freq_val_in = x_train_freq[tr_idx],  x_train_freq[val_idx]
        y_tr_in,    y_val_in    = y_train[tr_idx],        y_train[val_idx]

        # ── Fit normalization ONLY on the inner training split ────────
        def normalize(x, mn, mx):
            scale = np.where(mx - mn == 0, 1.0, mx - mn)
            return (x - mn) / scale

        ts_min = np.nanmin(x_tr_in,    axis=0);  ts_max = np.nanmax(x_tr_in,    axis=0)
        la_min = np.nanmin(last_tr_in, axis=0);  la_max = np.nanmax(last_tr_in, axis=0)
        y_min  = y_tr_in.min();                  y_max  = y_tr_in.max()

        x_tr_in     = normalize(x_tr_in,     ts_min, ts_max)
        x_val_in    = normalize(x_val_in,    ts_min, ts_max)
        x_te_norm   = normalize(x_test,      ts_min, ts_max)

        last_tr_in  = normalize(last_tr_in,  la_min, la_max)
        last_val_in = normalize(last_val_in, la_min, la_max)
        x_test_last = normalize(x_test_last, la_min, la_max)

        y_tr_in  #= normalize(y_tr_in,  y_min, y_max)
        y_val_in #= normalize(y_val_in, y_min, y_max)
        y_test   #= normalize(y_test,   y_min, y_max)

        # ── Build loaders ─────────────────────────────────────────────
        train_loader, _ = dataloaders_double_cv_cohort(
            x_tr_in, t_tr_in, last_tr_in, freq_tr_in, y_tr_in,
            x_tr_in, t_tr_in, last_tr_in, freq_tr_in, y_tr_in
        )

        _, val_loader = dataloaders_double_cv_cohort(
            x_tr_in,  t_tr_in,  last_tr_in,  freq_tr_in,  y_tr_in,
            x_val_in, t_val_in, last_val_in, freq_val_in, y_val_in
        )

        _, test_loader = dataloaders_double_cv_cohort(
            x_tr_in,   t_tr_in,   last_tr_in,  freq_tr_in,  y_tr_in,
            x_te_norm, x_test_t, x_test_last, x_test_freq, y_test
        )

        training_data.append(train_loader)
        validation_data.append(val_loader)
        test_data_loader.append(test_loader)

    return test_data_loader, training_data, validation_data, x_tr_in.shape, ts_min, ts_max, y_min, y_max
    

In [20]:
test_Dataloader, train_Dataloader, valid_Dataloader, data_shape, ts_min, ts_max, y_min , y_max = cv_fold_splits_cohorts(temporal_data_features,
                                                                                                                         temporal_timedelta,
                                                                                                                         last_obs_data, 
                                                                                                                         new_arr,icu_mor,
                                                                                                                         n_fold=10)

<------- OUTER FOLD 1 ------->
<------- OUTER FOLD 2 ------->
<------- OUTER FOLD 3 ------->
<------- OUTER FOLD 4 ------->
<------- OUTER FOLD 5 ------->
<------- OUTER FOLD 6 ------->
<------- OUTER FOLD 7 ------->
<------- OUTER FOLD 8 ------->
<------- OUTER FOLD 9 ------->
<------- OUTER FOLD 10 ------->


In [ ]:
ts_min, ts_max, y_min , y_max

In [21]:
dn=f"Documents/MIMICIII"
if not os.path.exists(dn):
    os.makedirs(dn)

In [ ]:
seq_length = temporal_data_features.shape[1]
input_dim = temporal_data_features.shape[-1]
hidden_dim, output_dim  = 64, icu_mor.shape[-1]
seq_length, input_dim, hidden_dim, output_dim, data_shape
seq_length, input_dim, hidden_dim, output_dim, data_shape
taskname=f"PHYSIONET2012_ICU_FIRST_ALL_SUBJECTS_{seq_length}_HOURS_DATA_10_FOLDS_DyFAIP_FREQS_SET_ZERO".upper()
task=f"{os.path.join(dn, f'{taskname}')}"
if not os.path.exists(task):
    os.makedirs(task)
task

In [23]:
icu_mor.max()

np.int64(1)

In [24]:
train_val_loader = random.sample(list(zip(train_Dataloader, valid_Dataloader)), len(train_Dataloader))
np.savez(os.path.join(task, f"train_test_data.npz"), 
            folds_data_test= test_Dataloader,
            folds_data_train_valid= train_val_loader,)
        
np.savez(os.path.join(task, f"data_max_min.npz"), 
         data_max=ts_max, data_min=ts_min,
         data_targets_max=icu_mor.max(), data_targets_min=icu_mor.min(),
         shape_data=data_shape,seq_length=seq_length,
         input_dim=input_dim, output_dim=output_dim)
input_dim

28

In [ ]:
ts_min, ts_max, y_min , y_max

In [ ]:
unique, counts = np.unique(icu_mor, return_counts=True)

dict(zip(unique, counts))

In [ ]:
forward["imputations"].shape

In [ ]:
class BRITS_I(nn.Module):
    def __init__(self, inputs_size):
        super(BRITS_I, self).__init__()
        self.inputs_size=inputs_size
        self.build()

    def build(self):
        self.rits_f = brits_i(self.inputs_size)
        self.rits_b = brits_i(self.inputs_size)

    def forward(self, inputs_data, masks_data, delta_data):
        ret_f = self.rits_f(inputs_data, masks_data, delta_data)
        ret_b = self.rits_b(torch.flip(inputs_data, [1]),torch.flip(masks_data, [1]),
                            torch.flip(delta_data, [1]))

        ret = self.merge_ret(ret_f, ret_b)

        return ret

    def merge_ret(self, ret_f, ret_b):
        loss_f = ret_f['loss']
        loss_b = ret_b['loss']
        loss_c = self.get_consistency_loss(ret_f['imputations'], ret_b['imputations'])

        loss = loss_f + loss_b + loss_c

        predictions = (ret_f['predictions'] + ret_b['predictions']) / 2
        imputations = (ret_f['imputations'] + ret_b['imputations']) / 2

        ret_f['loss'] = loss
        ret_f['predictions'] = predictions
        ret_f['imputations'] = imputations

        return ret_f

    def get_consistency_loss(self, pred_f, pred_b):
        loss = torch.pow(pred_f - pred_b, 2.0).mean()
        return loss


In [ ]:
labels= torch.zeros([4,1])
is_train= torch.zeros([4,1])

In [ ]:
device = torch.device("cpu" if torch.cuda.is_available() else "cpu")

rits_f = brits_i(x1.shape[-1]).to(device)
rits_b = brits_i(x1.shape[-1]).to(device)
rits_f

ret_f = rits_f(x1, x_mask_t, x2)
ret_b = rits_b(torch.flip(x1, [1]), 
               torch.flip(x_mask_t, [1]),
               torch.flip(x2, [1]))

In [ ]:
brits = BRITS_I(x1.shape[-1]).to(device)
brits

In [ ]:
outs = brits(x1, x_mask_t, x2)

In [ ]:
outs

In [ ]:
outs['loss']

In [ ]:
x1

In [ ]:
features_notes = ["SUBJECT_ID", "HADM_ID", "ICUSTAY_ID", 'PRESENT_ILLNESS',
                  'MEDICAL_HISTORY', 'PERTINENT_RESULTS', 'HOSPITAL_COURSE']

# Baseline methods

In [ ]:
import fancyimpute

In [ ]:
x_mask_t=1-torch.isnan(x1).to(torch.float32)

In [ ]:
x_mask_t

In [ ]:
x1[torch.isnan(x1)] = 0 
x1[torch.isinf(x1)] = 0

In [ ]:
# Algo1: Mean imputation

X_mean = []

print(len(x1))

for x, y in zip(x1, labels):
    X_mean.append(fancyimpute.SimpleFill().fill(x, x_mask_t))


In [ ]:
x

In [ ]:
for xi, y in zip(x1, labels):
    print(xi.shape)

In [ ]:
torch.randn(4, 48, 103)

In [ ]:
x1

In [ ]:
x1.shape

In [ ]:
x_mask_x=torch.isnan(x1).to(torch.float32)

In [ ]:
x_mask_x

In [ ]:
x1

In [ ]:
# Algo2: KNN imputation

X_knn = []

for xi, y in zip(x1, labels):
    X_knn.append(fancyimpute.KNN(k=10, verbose=False).fit_transform(xi))

In [ ]:
x1

In [ ]:
x1 = x1.view(x1.size(0), -1).numpy()

In [ ]:
x1

In [ ]:
from fancyimpute import MatrixFactorization

In [ ]:
# Note: Ensure missing values are represented as np.nan in tensor_reshaped
imputed_array = MatrixFactorization().fit_transform(x1)

In [ ]:
imputed_array

In [ ]:
x1.shape

In [ ]:
import torch
import numpy as np
from fancyimpute import MatrixFactorization

# Assuming `tensor` is your input data tensor with shape torch.Size([4, 48, 103])
tensor = torch.randn(4, 48, 103)  # Example tensor

# Step 1: Reshape your tensor to 2D
tensor_reshaped = tensor.view(tensor.size(0), -1).numpy()  # Converts to NumPy array as well

# Steps 2 & 3: Impute missing values
# Note: Ensure missing values are represented as np.nan in tensor_reshaped
imputed_array = MatrixFactorization().fit_transform(tensor_reshaped)

# Step 4: Reshape back to the original shape
imputed_tensor_reshaped = imputed_array.reshape((4, 48, 103))

# Step 5: Convert back to a PyTorch tensor (if necessary)
imputed_tensor = torch.tensor(imputed_tensor_reshaped, dtype=tensor.dtype)

# `imputed_tensor` now contains the imputed data with the original shape


In [ ]:
n = len(x1)
batch_size = 128
nb_batch = (n + batch_size - 1) // batch_size

for xi, y in zip(x1, labels):
    #y = np.concatenate(labels[i * batch_size: (i + 1) * batch_size])
    x_mice = fancyimpute.IterativeImputer.fit_transform(xi)

    X_mice.append(x_mice)

In [ ]:
fancyimpute.KNN(k=10, verbose=False).fit_transform

In [ ]:
from fancyimpute import IterativeImputer as MICE

In [ ]:
fancyimpute.IterativeImputer.fit_transform

In [ ]:
X_mice = np.concatenate(X_mice, axis=0)

In [ ]:
X_mf = []

for i, (x_j, y) in enumerate(zip(x1, labels)):
    X_mf.append(fancyimpute.MatrixFactorization(verbose=False).fit_transform(x_j))

In [ ]:
X_mf = np.stack(X_mf, axis=0)

In [ ]:
X_mf

In [ ]:
X_knn

In [ ]:
X_knn = np.stack(X_knn, axis=0)

In [ ]:
X_knn

In [ ]:
X_knn.shape

In [ ]:
X_knn.append(fancyimpute.KNN(k=10, verbose=False).prepare_input_data

In [ ]:
fancyimpute.KNN

In [ ]:

X_c = np.concatenate(X, axis=0).reshape(-1, 48, 35)
Y_c = np.concatenate(Y, axis=0).reshape(-1, 48, 35)
Z_c = np.array(Z)
X_mean = np.concatenate(X_mean, axis=0).reshape(-1, 48, 35)

temp_decay = TemporalDecay(input_size = x1.shape[-1])
regression = nn.Linear(RNN_HID_SIZE, x1.shape[-1])
rnn_cell = nn.LSTMCell(x1.shape[-1] * 2, RNN_HID_SIZE)
out = nn.Linear(RNN_HID_SIZE, 1)
h = Variable(torch.zeros((x1.size()[0], RNN_HID_SIZE)))
c = Variable(torch.zeros((x1.size()[0], RNN_HID_SIZE)))
x_loss = 0.0
imputations = []
for t in range(SEQ_LEN):
    x = x1[:, t, :]
    m = x_mask_t[:, t, :]
    d = x2[:, t, :]
    gamma = temp_decay(d)
    h = h * gamma
    x_h = regression(h)
    x_c =  m * x +  (1 - m) * x_h
    x_loss += torch.sum(torch.abs(x - x_h) * m) / (torch.sum(m) + 1e-5)
    inputs = torch.cat([x_c, m], dim = 1)
    h, c = rnn_cell(inputs, (h, c))
    imputations.append(x_c.unsqueeze(dim = 1))
    print(h.shape, x_h.shape, x.shape, x_c.shape, inputs.shape)
    #print()
    print(m * x)
imputations = torch.cat(imputations, dim = 1)
y_h = out(h)
y_h = torch.sigmoid(y_h)

# GRU-D- Generate-sample-data

In [ ]:
n_sample = 100
n_dim = 7
n_max_timestamp = 17
n_class = 3

In [ ]:
inputs = np.empty(shape=(n_sample), dtype=object)
masking =  np.empty(shape=(n_sample), dtype=object)
timestamp = np.empty(shape=(n_sample), dtype=object)
label_taskname = np.empty(shape=(n_sample, n_class), dtype=int)
print(label_taskname.shape, timestamp.shape)

In [ ]:
label_taskname = np.stack((
    np.random.binomial(1, 0.3, size=(n_sample)), 
    np.random.binomial(1, 0.6, size=(n_sample)), 
    np.random.binomial(1, 0.2, size=(n_sample))
), axis=-1)
print(label_taskname.shape)

In [ ]:
inputs

In [ ]:
timestamp

In [ ]:
for i in range(n_sample):
    len_t_i = np.random.randint(5, 17)
    timestamp_i = np.random.random(size=(len_t_i)) * 10 + 1
    timestamp_i = np.cumsum(timestamp_i) - timestamp_i[0]
    timestamp[i] = timestamp_i
print(timestamp.shape)
print(timestamp[0].shape)

In [ ]:
timestamp.shape

In [ ]:
timestamp

In [ ]:
inputs

In [ ]:
for i in range(n_sample):
    start = np.random.random(size=n_dim)*np.pi*2
    input_i = np.zeros(shape=(n_dim, len(timestamp[i])), dtype=float)
    if label_taskname[i][0]:
        input_i += np.cos(start[:, np.newaxis] + timestamp[i][np.newaxis, :])
    if label_taskname[i][1]:
        input_i += np.cos(2 * (start[:, np.newaxis] + timestamp[i][np.newaxis, :])) + 1
    if label_taskname[i][2]:
        input_i += np.cos(5 * (start[:, np.newaxis] + timestamp[i][np.newaxis, :])) + 2
    inputs[i] = input_i.T

print(inputs.shape)
print(inputs[0].shape)

In [ ]:
inputs[1].shape

In [ ]:
inputs

In [ ]:
for i in range(n_sample):
    masking_i = (np.random.random(size=(len(timestamp[i]), n_dim)) > 0.7).astype(int)
    masking[i] = masking_i
    inputs[i][masking_i == 0] = np.nan

print(masking.shape)
print(masking[0].shape)
print(masking[0][0])
print(inputs[0][0])

In [ ]:
inputs

In [ ]:
inputs

In [ ]:
n_split = 5
import numpy as np
import os
from sklearn import model_selection 

In [ ]:
fold_taskname = np.empty(shape=(n_split, 3), dtype=object)

idx_all = sorted(range(100))
for i_split, idx in enumerate(model_selection.KFold(5, shuffle=False).split(idx_all)):
    fold_taskname[i_split][2] = idx[-1]
for i_split in range(n_split):
    fold_taskname[i_split][1] = fold_taskname[(i_split + 1) % n_split][2]
    print(fold_taskname[i_split][1])
    fold_taskname[i_split][0] = np.asarray(sorted(set(idx_all).difference(fold_taskname[i_split][1]).difference(fold_taskname[i_split][2])))

print(fold_taskname[0][0].shape, fold_taskname[0][1].shape, fold_taskname[0][2].shape)



In [ ]:
fold_taskname

In [ ]:
inputs

In [ ]:
mean_taskname = np.zeros((n_split, 3, n_dim)) * np.nan
std_taskname = np.zeros((n_split, 3, n_dim)) * np.nan
for i_split in range(n_split):
    x_tr = np.concatenate(inputs[fold_taskname[i_split][0]], axis=0)
    mean_taskname[i_split][0] = np.nanmean(x_tr, axis=0)
    std_taskname[i_split][0] = np.nanstd(x_tr, axis=0)
    
print(mean_taskname[0][0])
print(std_taskname[0][0])

In [ ]:
mean_taskname.shape, inputs.shape

In [ ]:
mean_taskname

In [ ]:
std_taskname

In [ ]:
class AMITA2i_LSTM_e(torch.jit.ScriptModule):
    def __init__(self, input_size, hidden_size,seq_len, output_dim, batch_first=True, bidirectional=True):
        super(AMITA2i_LSTM_e, self).__init__()
        self.input_size = input_size
        self.output_dim = output_dim
        self.initializer_range=0.02
        self.hidden_size = hidden_size
        self.seq_len = seq_len
        self.batch_first = batch_first
        self.bidirectional = bidirectional
        self.c1 = torch.Tensor([1]).float()
        self.c2 = torch.Tensor([np.e]).float()
        self.ones = torch.ones([self.input_size,1, self.hidden_size]).float()
        self.decay_features = torch.Tensor(torch.arange(self.input_size)).float()
        self.register_buffer('c1_const', self.c1)
        self.register_buffer('c2_const', self.c2)
        self.register_buffer("ones_const", self.ones)
        self.alpha = torch.FloatTensor([0.5])
        self.imp_weight = torch.FloatTensor([0.05])
        self.alpha_imp = torch.FloatTensor([0.5])
        self.register_buffer("factor", self.alpha)
        self.register_buffer("imp_weight_freq", self.imp_weight)
        self.register_buffer("features_decay", self.decay_features)
        self.register_buffer("factor_impu", self.alpha_imp)
        
        self.U_j = nn.Parameter(torch.normal(0.0, self.initializer_range, size=(self.input_size, 1, self.hidden_size)))
        self.U_i = nn.Parameter(torch.normal(0.0, self.initializer_range, size=(self.input_size, 1, self.hidden_size)))
        self.U_f = nn.Parameter(torch.normal(0.0, self.initializer_range, size=(self.input_size, 1, self.hidden_size)))
        self.U_o = nn.Parameter(torch.normal(0.0, self.initializer_range, size=(self.input_size, 1, self.hidden_size)))
        self.U_c = nn.Parameter(torch.normal(0.0, self.initializer_range, size=(self.input_size, 1, self.hidden_size)))
        self.U_last = nn.Parameter(torch.normal(0.0, self.initializer_range, size=(self.input_size, 1, self.hidden_size)))
        self.U_time = nn.Parameter(torch.normal(0.0, self.initializer_range, size=(self.input_size, 1, self.hidden_size)))
        self.Dw = nn.Parameter(torch.normal(0.0, self.initializer_range, size=(self.input_size, 1, self.hidden_size)))
        
        self.W_j = nn.Parameter(torch.normal(0.0, self.initializer_range, size=(self.input_size, self.hidden_size, self.hidden_size)))
        self.W_i = nn.Parameter(torch.normal(0.0, self.initializer_range, size=(self.input_size, self.hidden_size, self.hidden_size)))
        self.W_f = nn.Parameter(torch.normal(0.0, self.initializer_range, size=(self.input_size, self.hidden_size, self.hidden_size)))
        self.W_o = nn.Parameter(torch.normal(0.0, self.initializer_range, size=(self.input_size, self.hidden_size, self.hidden_size)))
        self.W_c = nn.Parameter(torch.normal(0.0, self.initializer_range, size=(self.input_size, self.hidden_size, self.hidden_size)))
        self.W_d = nn.Parameter(torch.normal(0.0, self.initializer_range, size=(self.input_size, self.hidden_size, self.hidden_size)))
        self.W_decomp = nn.Parameter(torch.normal(0.0, self.initializer_range, size=(self.input_size, self.hidden_size, self.hidden_size)))
        
        self.W_cell_i = nn.Parameter(torch.normal(0.0, self.initializer_range, size=(self.input_size, self.hidden_size)))
        self.W_cell_f= nn.Parameter(torch.normal(0.0, self.initializer_range, size=(self.input_size, self.hidden_size)))
        self.W_cell_o = nn.Parameter(torch.normal(0.0, self.initializer_range, size=(self.input_size, self.hidden_size)))
        
        
        self.b_decomp = nn.Parameter(torch.normal(0.0, self.initializer_range, size=(self.input_size, self.hidden_size)))
        self.b_j = nn.Parameter(torch.normal(0.0, self.initializer_range, size=(self.input_size, self.hidden_size)))
        self.b_i = nn.Parameter(torch.normal(0.0, self.initializer_range, size=(self.input_size, self.hidden_size)))
        self.b_f = nn.Parameter(torch.normal(0.0, self.initializer_range, size=(self.input_size, self.hidden_size)))
        self.b_o = nn.Parameter(torch.normal(0.0, self.initializer_range, size=(self.input_size, self.hidden_size)))
        self.b_c = nn.Parameter(torch.normal(0.0, self.initializer_range, size=(self.input_size, self.hidden_size)))
        self.b_last = nn.Parameter(torch.normal(0.0, self.initializer_range, size=(self.input_size, self.hidden_size)))
        self.b_time = nn.Parameter(torch.normal(0.0, self.initializer_range, size=(self.input_size, self.hidden_size)))
        self.b_d = nn.Parameter(torch.normal(0.0, self.initializer_range, size=(self.input_size, self.hidden_size)))
        # Interpolation
        self.W_ht_mask = nn.Parameter(torch.normal(0.0, self.initializer_range, size=(self.input_size, self.hidden_size,1)))
        self.W_ct_mask = nn.Parameter(torch.normal(0.0, self.initializer_range, size=(self.input_size, self.hidden_size,1)))
        self.b_j_mask = nn.Parameter(torch.normal(0.0, self.initializer_range, size=(self.input_size, self.hidden_size)))
        
        self.W_ht_last = nn.Parameter(torch.normal(0.0, self.initializer_range, size=(self.input_size, self.hidden_size,1)))
        self.W_ct_last = nn.Parameter(torch.normal(0.0, self.initializer_range, size=(self.input_size, self.hidden_size,1)))
        self.b_j_last = nn.Parameter(torch.normal(0.0, self.initializer_range, size=(self.input_size, 1))) 
        
        #Gate Linear Unit for last records
        self.activation_layer = nn.ELU()
        
        self.F_alpha_n = nn.Parameter(torch.normal(0.0, self.initializer_range, size=(self.input_size,self.hidden_size*2, 1)))
        self.F_alpha_n_b = nn.Parameter(torch.normal(0.0, self.initializer_range, size=(self.input_size,1)))
        self.F_beta = nn.Linear(4*self.hidden_size, 1)
        self.Phi = nn.Linear(4*self.hidden_size, self.output_dim)
    @torch.jit.script_method    
    def TLSTM_unit(self, prev_hidden_memory, cell_hidden_memory, inputs, times, last_data, freq_list):
        h_tilda_t, c_tilda_t = prev_hidden_memory, cell_hidden_memory,
        x = inputs
        t = times
        l = last_data
        freq=freq_list
        T = self.map_elapse_time(t)
        # Short-term memory contribution
        D_ST = torch.tanh(torch.einsum("bij,ijk->bik", c_tilda_t, self.W_decomp))  
        # Apply temporal decay to D-STM
        decay_factor = torch.mul(T, self.freq_decay(freq, h_tilda_t))
        D_ST_decayed = D_ST * decay_factor
        # Long-term memory contribution
        LTM = c_tilda_t - D_ST + D_ST_decayed  
        # Combine short-term and long-term memory
        c_tilda_t = D_ST_decayed + LTM
        #frequency weights for imputation of missing data based on frequencies of features
        #factor_feature = torch.div(torch.exp(-self.factor_impu *freq),torch.exp(-self.factor_impu *freq).max()).unsqueeze(1)
        factor_feature = (self.seq_len-freq)*torch.exp(-self.imp_weight_freq * freq)

        factor_imp = torch.div(torch.exp(self.factor_impu *freq), torch.exp(self.factor_impu *freq).max()).unsqueeze(1)
        frequencies = (self.seq_len-freq)*torch.exp(-self.factor * self.features_decay)
        frequencies = torch.div(frequencies,frequencies.max()).unsqueeze(-1)
        # Imputation gate for inputs x last records
        x_last_hidden =torch.tanh(torch.einsum("bij,ijk->bik", self.freq_decay(freq, h_tilda_t),self.W_ht_last)+\
                                  torch.einsum("bij,ijk->bik", self.freq_decay(freq, c_tilda_t),
                                  self.W_ct_last)+self.b_j_last).permute(0, 2, 1)
        
        imputat_imputs = torch.tanh(torch.einsum("bij,ijk->bik", self.freq_decay(freq, h_tilda_t), self.W_ht_mask)+\
                                    torch.einsum("bij,ijk->bik",self.freq_decay(freq, c_tilda_t),self.W_ct_mask)+\
                                    self.b_j_mask).permute(0, 2, 1)
        # Replace nan data with the impuated value generated from LSTM memory and frequencies weights
        comp_missed_last = torch.where(factor_imp==factor_imp.max(), frequencies.permute(0,2,1)*x_last_hidden, 
                                  factor_feature.unsqueeze(1)*x_last_hidden)
        comp_missed_x = torch.where(factor_imp==factor_imp.max(), frequencies.permute(0,2,1)*imputat_imputs,
                                    factor_feature.unsqueeze(1)*imputat_imputs)
        
        x_last = torch.where(torch.isnan(l.unsqueeze(1)), comp_missed_last, l.unsqueeze(1))
        # Ajust previous to incoporate the latest records for each feature
        last_tilda_t = self.activation_layer(torch.einsum("bij,jik->bjk", x_last, self.U_last)+self.b_last)
        h_tilda_t = h_tilda_t + last_tilda_t
        
        #print(x.unsqueeze(1).shape, comp_missed_x.shape)
        imputed_x = torch.where(torch.isnan(x.unsqueeze(1)), comp_missed_x, x.unsqueeze(1))
        print(imputed_x.shape, x.unsqueeze(1).shape, comp_missed_x.shape, torch.mean(comp_missed_x, dim=1, keepdim=True).shape)
        # Capturing Temporal Dependencies wrt to the previous hidden state
        j_tilda_t = torch.tanh(torch.einsum("bij,ijk->bik", h_tilda_t, self.W_j) +\
                               torch.einsum("bij,jik->bjk", imputed_x,self.U_j) + self.b_j)
        
        # Time Gate
        t_gate = torch.sigmoid(torch.einsum("bij,jik->bjk",imputed_x, self.U_time) + 
                               torch.sigmoid(self.map_elapse_time(t)) + self.b_time)
        # Input Gate
        i= torch.sigmoid(torch.einsum("bij,jik->bjk",imputed_x, self.U_i)+\
                         torch.einsum("bij,ijk->bik", h_tilda_t, self.W_i)+\
                         c_tilda_t*self.W_cell_i + self.b_i*self.freq_decay(freq, j_tilda_t))
        # Forget Gate
        f= torch.sigmoid(torch.einsum("bij,jik->bjk", imputed_x, self.U_f)+\
                         torch.einsum("bij,ijk->bik", h_tilda_t, self.W_f)+\
                         c_tilda_t*self.W_cell_f + self.b_f+j_tilda_t)

        f_new = f * self.map_elapse_time(t) + (1 - f) *  self.freq_decay(freq, j_tilda_t)
        # Candidate Memory Cell
        C =torch.tanh(torch.einsum("bij,jik->bjk", imputed_x, self.U_c)+\
                      torch.einsum("bij,ijk->bik", h_tilda_t, self.W_c) + self.b_c)
        # Current Memory Cell
        Ct = (f_new + t_gate) * c_tilda_t + i * j_tilda_t * t_gate * C
        # Output Gate        
        o = torch.sigmoid(torch.einsum("bij,jik->bjk",imputed_x, self.U_o)+
                          torch.einsum("bij,ijk->bik", h_tilda_t, self.W_o)+
                          t_gate + last_tilda_t + Ct*self.W_cell_o + self.b_o)
        # Current Hidden State
        h_tilda_t = o * torch.tanh(Ct+last_tilda_t)
        
        return h_tilda_t, Ct, self.freq_decay(freq, j_tilda_t), f_new,imputed_x
    
    @torch.jit.script_method
    def map_elapse_time(self, t):
        T = torch.div(self.c1_const, torch.log(t + self.c2_const))
        T = torch.einsum("bij,jik->bjk", T.unsqueeze(1), self.ones_const)
        return T

    @torch.jit.script_method
    def freq_decay(self, freq_dict: torch.Tensor, ht: torch.Tensor):
        freq_weight = torch.exp(-self.factor_impu * freq_dict)
        weights = torch.sigmoid(torch.einsum("bij,jik->bjk",freq_weight.unsqueeze(-1),self.Dw)+\
                                torch.einsum("bij,ijk->bik", ht, self.W_d)+ self.b_d)
        return weights
    @torch.jit.script_method
    def forward(self, inputs, times, last_values, freqs):
        device = inputs.device
        if self.batch_first:
            batch_size = inputs.size()[0]
            inputs = inputs.permute(1, 0, 2)
            last_values = last_values.permute(1, 0, 2)
            freqs = freqs.permute(1, 0, 2)
            times = times.transpose(0, 1)
        else:
            batch_size = inputs.size()[1]
        prev_hidden = torch.zeros((batch_size, inputs.size()[2], self.hidden_size), device=device)
        prev_cell = torch.zeros((batch_size, inputs.size()[2], self.hidden_size), device=device)
       
        seq_len = inputs.size()[0]
        imputed_inputs = torch.jit.annotate(List[Tensor], [])
        hidden_his = torch.jit.annotate(List[Tensor], [])
        weights_decay = torch.jit.annotate(List[Tensor], [])
        weights_fgate = torch.jit.annotate(List[Tensor], [])
        for i in range(seq_len):
            prev_hidden, prev_cell,pre_we_decay, fgate_f, imputed_x = self.TLSTM_unit(prev_hidden,
                                                                           prev_cell, 
                                                                           inputs[i],
                                                                           times[i], 
                                                                           last_values[i],
                                                                           freqs[i])
            hidden_his += [prev_hidden]
            imputed_inputs += [imputed_x]
            weights_decay += [pre_we_decay]
            weights_fgate += [fgate_f]
        imputed_inputs = torch.stack(imputed_inputs)
        hidden_his = torch.stack(hidden_his)
        weights_decay = torch.stack(weights_decay)
        weights_fgate = torch.stack(weights_fgate)
        if self.bidirectional:
            second_hidden = torch.zeros((batch_size, inputs.size()[2], self.hidden_size), device=device)
            second_cell = torch.zeros((batch_size, inputs.size()[2], self.hidden_size), device=device)
            second_inputs = torch.flip(inputs, [0])
            second_times = torch.flip(times, [0])
            second_hidden_his = torch.jit.annotate(List[Tensor], [])
            second_weights_decay = torch.jit.annotate(List[Tensor], [])
            second_weights_fgate = torch.jit.annotate(List[Tensor], [])
            for i in range(seq_len):
                if i == 0:
                    time = times[i]
                else:
                    time = second_times[i-1]
                second_hidden, second_cell,b_we_decay,fgate_b,_ = self.TLSTM_unit(second_hidden,
                                                                                second_cell, 
                                                                                second_inputs[i],
                                                                                time,
                                                                                last_values[i],
                                                                                freqs[i])
                second_hidden_his += [second_hidden]
                second_weights_decay += [b_we_decay]
                second_weights_fgate += [fgate_b]
            second_hidden_his = torch.stack(second_hidden_his)
            second_weights_fgate = torch.stack(second_weights_fgate)
            second_weights_decay = torch.stack(second_weights_decay)
            weights_decay =torch.cat((weights_decay, second_weights_decay), dim=-1)
            weights_fgate =torch.cat((weights_fgate, second_weights_fgate), dim=-1)
            hidden_his = torch.cat((hidden_his, second_hidden_his), dim=-1)
            prev_hidden = torch.cat((prev_hidden, second_hidden), dim=-1)
            prev_cell = torch.cat((prev_cell, second_cell), dim=-1)
        if self.batch_first:
            hidden_his = hidden_his.permute(1, 0, 2, 3)
            weights_decay = weights_decay.permute(1, 0, 2, 3)
            weights_fgate = weights_fgate.permute(1, 0, 2, 3)
            
        alphas = torch.tanh(torch.einsum("btij,ijk->btik", hidden_his, self.F_alpha_n) + self.F_alpha_n_b)
        alphas = torch.exp(alphas)
        alphas = alphas/torch.sum(alphas, dim=1, keepdim=True)
        g_n = torch.sum(alphas*hidden_his, dim=1)
        hg = torch.cat([g_n, prev_hidden], dim=2)
        mu = self.Phi(hg)
        betas = torch.tanh(self.F_beta(hg))
        betas = torch.exp(betas)
        betas = betas/torch.mean(betas, dim=1, keepdim=True)
        #betas = betas/torch.max(betas, dim=1, keepdim=True).values
        mean = torch.sum(betas*mu, dim=1)
        return mean, alphas, betas , weights_decay, weights_fgate, imputed_inputs

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import random

In [ ]:
def create_porous_media(nx, ny, l, R, e=0.5, a=1.5, w=45, gamma=67):
    grains = []
    distances = []
    rcurv = []
    pressures = []

    for i in range(nx):
        for j in range(ny):
            x = i * l
            y = j * l
            r = random.uniform(0, e * (a - 1) * R)
            if r > 0:
                theta = random.uniform(0, 2 * np.pi)
                dx = r * np.cos(theta)
                dy = r * np.sin(theta)
                x += dx
                y += dy
            overlaps = [(grain[0] - x) ** 2 + (grain[1] - y) ** 2 < (grain[2] + R) ** 2 for grain in grains]
            while any(overlaps):
                r = random.uniform(0, e * (a - 1) * R)
                theta = random.uniform(0, 2 * np.pi)
                dx = r * np.cos(theta)
                dy = r * np.sin(theta)
                x += dx
                y += dy
                overlaps = [(grain[0] - x) ** 2 + (grain[1] - y) ** 2 < (grain[2] + R) ** 2 for grain in grains]
            grains.append((x, y, R))

    impermeable_boundaries = [(0, 1), (1, 2), (2, 3), (3, 4), (4, 5), 
                              (0, 6), (6, 12), (12, 18), (18, 24), (24, 30), 
                              (5, 11), (11, 17), (17, 23), (23, 29), (29, 35)]

    for i in range(nx * ny):
        if i % nx < nx - 1 and (i, i + 1) not in impermeable_boundaries:
            dx = grains[i][0] - grains[i + 1][0]
            dy = grains[i][1] - grains[i + 1][1]
            distance = np.sqrt(dx ** 2 + dy ** 2) - grains[i][2] - grains[i + 1][2]
            distances.append((i, i + 1, distance))
        if i // nx < ny - 1 and (i, i + nx) not in impermeable_boundaries:
            dx = grains[i][0] - grains[i + nx][0]
            dy = grains[i][1] - grains[i + nx][1]
            distance = np.sqrt(dx ** 2 + dy ** 2) - grains[i][2] - grains[i + nx][2]
            distances.append((i, i + nx, distance))

    for dist in distances:
        rcurv_value = (R + dist[2] / 2 - R * np.cos(np.radians(a))) / np.cos(np.radians(a + w))
        rcurv.append((dist[0], dist[1], rcurv_value))

    for rc in rcurv:
        pressure = gamma / rc[2]
        pressures.append((rc[0], rc[1], pressure))

    return grains, distances, rcurv, pressures

In [ ]:
nx, ny = 6, 6
R = 0.5
e = 1
a = 1.5
w = 45
gamma = 67
l = 2 * a * R
grains, distances, rcurv, pressures = create_porous_media(nx, ny, l, R, e, a, w, gamma)

In [ ]:
x_min, x_max = -1, nx * l
y_min, y_max = -1, ny * l
rectangle = plt.Rectangle((x_min, y_min), x_max - x_min, y_max - y_min, fill=False, edgecolor='black')
ax.add_patch(rectangle)

ax.set_xlim(x_min - 1, x_max + 1)
ax.set_ylim(y_min - 1, y_max + 1)
plt.show()

In [ ]:
print("Distances between neighboring grains:")
for distance in distances:
    print(f"Grains {distance[0]} and {distance[1]}: {distance[2]:.4f}")

print("rcurv values calculated from distances:")
for rc in rcurv:
    print(f"Grains {rc[0]} and {rc[1]}: {rc[2]:.4f}")

print("Pressures calculated from rcurv values:")
for pressure in pressures:
    print(f"Grains {pressure[0]} and {pressure[1]}: {pressure[2]:.4f}")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

class InvasionPercolationModel:
    def __init__(self, size, start_position):
        self.size = size
        self.depth_increment = 10
        # Initialize boundary pressures (horizontal and vertical)
        self.horiz_pressures = np.random.randint(80, 200, size=(size[0], size[1]-1)) + np.arange(size[0]).reshape(-1,1) * self.depth_increment
        self.vert_pressures = np.random.randint(80, 200, size=(size[0]-1, size[1])) + np.arange(size[0]-1).reshape(-1,1) * self.depth_increment
        self.visited = np.zeros(size, dtype=bool)  # Keep track of visited cells
        self.invaded_cells = [start_position]
        self.invasion_order = [0]  # Keep track of the order of invaded cells
        self.frontier = set()  # Store boundaries as ((x1, y1), (x2, y2))
        self.initialize_frontier(start_position)
        self.stop_invasion = False  # Flag to stop invasion if any cell in the last row is invaded
        self.last_row_depth = size[0] * self.depth_increment
        
    def initialize_frontier(self, start_position):
        x, y = start_position
        self.visited[x, y] = True
        # Add initial boundaries to the frontier
        if y > 0:
            self.frontier.add(((x, y-1), (x, y)))
        if y < self.size[1] - 1:
            self.frontier.add(((x, y), (x, y+1)))
        if x > 0:
            self.frontier.add(((x-1, y), (x, y)))
        if x < self.size[0] - 1:
            self.frontier.add(((x, y), (x+1, y)))

    def find_lowest_pressure_boundary(self):
        lowest_pressure = float('inf')
        lowest_boundary = None
        for boundary in self.frontier:
            (x1, y1), (x2, y2) = boundary
            # Determine if the boundary is horizontal or vertical
            if x1 == x2:  # Horizontal boundary
                pressure = self.horiz_pressures[x1, min(y1, y2)] 
            else:  # Vertical boundary
                pressure = self.vert_pressures[min(x1, x2), y1]
            if pressure < lowest_pressure:
                lowest_pressure = pressure
                lowest_boundary = boundary
        return lowest_boundary

    def invade_one_step(self):
        if not self.frontier or self.stop_invasion:
            return False
        lowest_boundary = self.find_lowest_pressure_boundary()
        if lowest_boundary is None:
            return False
        (x1, y1), (x2, y2) = lowest_boundary
        # Determine the new cell to invade
        new_cell = (x2, y2) if not self.visited[x2, y2] else (x1, y1)
        if self.visited[new_cell[0], new_cell[1]]:  # Skip already invaded cells
            self.frontier.remove(lowest_boundary)
            return True
        self.visited[new_cell[0], new_cell[1]] = True
        self.invaded_cells.append(new_cell)
        self.invasion_order.append(max(self.invasion_order) + 1)
        self.frontier.remove(lowest_boundary)
        self.update_frontier(new_cell)
        # Check if any cell in the last row is invaded
        if np.any(self.visited[-1]):
            self.stop_invasion = True
        return True

    def invade(self):
        while self.invade_one_step():
            pass

    def update_frontier(self, new_cell):
        x, y = new_cell
        # Add new boundaries to the frontier
        if y > 0 and not self.visited[x, y-1]:
            self.frontier.add(((x, y-1), (x, y)))
        if y < self.size[1] - 1 and not self.visited[x, y+1]:
            self.frontier.add(((x, y), (x, y+1)))
        if x > 0 and not self.visited[x-1, y]:
            self.frontier.add(((x-1, y), (x, y)))
        if x < self.size[0] - 1 and not self.visited[x+1, y]:
            self.frontier.add(((x, y), (x+1, y)))

# Example usage
size = (5, 5)  # Grid size
start_position = (0, 2)  # Starting position

model = InvasionPercolationModel(size, start_position)

# Invade cells
while model.invade_one_step():
    pass

# Visualize invaded cells
plt.imshow(model.visited, cmap='binary', interpolation='nearest')
plt.title('Invasion Visualization')
plt.show()

print(model.vert_pressures)
print(model.horiz_pressures)
print(model.visited)

# Output the order of invaded cells
print("Order of invaded cells:")
for cell, order in zip(model.invaded_cells, model.invasion_order):
    print(f"Cell {cell} invaded at order {order}")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import heapq

def initialize_grid(size):
    # Create a grid with random threshold values between 0 and 1
    grid = np.random.rand(size, size)
    return grid

def get_neighbors(x, y, size):
    # Get the neighboring cells (4-connectivity)
    neighbors = []
    if x > 0:
        neighbors.append((x-1, y))
    if x < size - 1:
        neighbors.append((x+1, y))
    if y > 0:
        neighbors.append((x, y-1))
    if y < size - 1:
        neighbors.append((x, y+1))
    return neighbors

def invasion_percolation(size, start):
    grid = initialize_grid(size)
    invaded = np.zeros_like(grid, dtype=bool)
    
    # Priority queue to keep track of the invasion front
    front = []
    heapq.heappush(front, (grid[start], start))
    
    while front:
        value, (x, y) = heapq.heappop(front)
        
        if invaded[x, y]:
            continue
        
        invaded[x, y] = True
        
        for nx, ny in get_neighbors(x, y, size):
            if not invaded[nx, ny]:
                heapq.heappush(front, (grid[nx, ny], (nx, ny)))
    
    return invaded, grid

def plot_invasion(invaded, grid):
    plt.figure(figsize=(10, 10))
    plt.imshow(grid, cmap='gray', alpha=0.5)
    plt.imshow(invaded, cmap='viridis', alpha=0.7)
    plt.colorbar()
    plt.title('Invasion Percolation')
    plt.show()

def main():
    size = 100  # Grid size
    start = (size // 2, size // 2)  # Starting point at the center
    
    invaded, grid = invasion_percolation(size, start)
    plot_invasion(invaded, grid)

if __name__ == "__main__":
    main()


In [ ]:
def metrics_binary(targets, predicted):
    # Calculate ROC curve and AUC for each class
    fpr = dict()
    tpr = dict()
    roc_auc = dict()
    scores = []
    for y_true, y_pred, in zip(targets, predicted):
        fpr, tpr, thresholds = metrics.roc_curve(y_pred, y_true)
        auc_score = metrics.auc(fpr, tpr)
        pr_score = metrics.average_precision_score(y_pred, y_true, average='macro')
        scores.append([np.round(np.mean(auc_score), 4),
                           np.round(np.mean(pr_score), 4)])
    return scores

In [ ]:
true_labels = np.array([
    [1, 0, 1],
    [0, 1, 0],
    [1, 1, 1],
    [0, 0, 1],
    [1, 0, 0]
])

# Example predicted scores (probability matrix of shape [n_samples, n_classes])
predicted_scores = np.array([
    [0.8, 0.2, 0.6],
    [0.4, 0.9, 0.3],
    [0.7, 0.8, 0.5],
    [0.3, 0.2, 0.8],
    [0.6, 0.1, 0.4]
])
predicted_scores1 = np.array([
    [0.008, 0.2, 0.6],
    [0.00840315, 0.00885529, 0.00982861],
    [0.01004217, 0.01073314, 0.01352474],
    [0.3, 0.2, 0.8],
    [0.02624058, 0.02606224, 0.02785756]
])

In [ ]:
import numpy as np
from sklearn.metrics import roc_curve, auc, average_precision_score, f1_score
def metrics_binary(targets, predicted_scores):
    # Calculate ROC curve and AUC for each class
    fpr = dict()
    tpr = dict()
    roc_auc = dict()
    scores = []
    for i in range(targets.shape[1]):
        fpr[i], tpr[i], _ = roc_curve(targets[:, i], predicted_scores[:, i])
        roc_auc[i] = auc(fpr[i], tpr[i])
    average_auc = np.mean(list(roc_auc.values()))
    pr_score = average_precision_score(targets, predicted_scores, average='macro')
    scores.append([np.round(average_auc, 4),
                  np.round(np.mean(pr_score), 4)])
    return scores

In [ ]:
def best_threshold(train_preds,y_true):
    delta, tmp = 0, [0, 0, 0]  # idx, cur, max
    for tmp[0] in np.arange(0.1, 1.01, 0.01):
        tmp[1] = f1_score(y_true, np.array(train_preds) > tmp[0], average='macro')
        if tmp[1] > tmp[2]:
            delta = tmp[0]
            tmp[2] = tmp[1]
    print('best threshold is {:.2f} with F1 score: {:.4f}'.format(delta, tmp[2]))
    return delta, tmp[2]

In [ ]:
loss = criterion(outputs, labels.float())

In [ ]:
def best_threshold(train_preds,y_true):
    delta, tmp = 0, [0, 0, 0]  # idx, cur, max
    for tmp[0] in np.arange(0.1, 1.01, 0.01):
        tmp[1] = f1_score(y_true, np.array(train_preds) > tmp[0], average='macro')
        if tmp[1] > tmp[2]:
            delta = tmp[0]
            tmp[2] = tmp[1]
    print('best threshold is {:.2f} with F1 score: {:.4f}'.format(delta, tmp[2]))
    return delta, tmp[2]

In [ ]:
delta, f1_scr = best_threshold(true_labels, predicted_scores)

In [ ]:
true_labels

In [ ]:
[[0.00397751 0.00416689 0.00378013 ... 0.02341746 0.02530959 0.02335033]
 [0.02422174 0.02557497 0.02881676 ... 0.08494653 0.09238774 0.08095301]
 [0.06780463 0.07040504 0.07879959 ... 0.16238336 0.17222995 0.16793032]
 ...
 [0.01722173 0.01967843 0.0269882  ... 0.13449737 0.13197243 0.13397935]
 [0.00410762 0.00497929 0.00380312 ... 0.01731556 0.01782284 0.01851909]
 [0.00421645 0.00455451 0.00575723 ... 0.05083179 0.05199789 0.05410205]]

In [ ]:
predicted_scores1

In [ ]:
print(sigmoid_v(predicted_scores1))

In [ ]:
import numpy as np
import math

# custom function
def sigmoid(x):
    return 1 / (1 + math.exp(-x))

# define vectorized sigmoid
sigmoid_v = np.vectorize(sigmoid)

# test
scores = np.array([ -0.54761371,  17.04850603,   4.86054302])
print(sigmoid_v(scores))


In [ ]:
scores = metrics_binary(true_labels,predicted_scores1,  )

In [ ]:
true_labels.shape

In [ ]:
scores

In [ ]:
import numpy as np
from sklearn.metrics import average_precision_score

# Example ground truth labels and predicted scores for a multi-label classification problem
# Assume we have 3 labels
y_true = np.array([
    [1, 0, 1],
    [0, 1, 0],
    [1, 1, 1],
    [0, 0, 1]
])

y_scores = np.array([
    [0.8, 0.1, 0.9],
    [0.2, 0.9, 0.4],
    [0.9, 0.8, 0.8],
    [0.1, 0.3, 0.7]
])

# Calculate the average precision score
average_precision = average_precision_score(y_true, y_scores, average="micro")

print("Average Precision Score: {:.4f}".format(average_precision))


In [ ]:
import numpy as np
import torch
from sklearn.metrics import precision_score, recall_score
from torch.utils.data import DataLoader, Dataset

# Dummy dataset
class MultiLabelDataset(Dataset):
    def __init__(self, num_samples=1000, num_classes=10):
        self.data = torch.randn(num_samples, num_classes)
        self.labels = (torch.rand(num_samples, num_classes) > 0.5).float()

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.data[idx], self.labels[idx]

# Dummy model
class MultiLabelModel(torch.nn.Module):
    def __init__(self, num_classes=10):
        super(MultiLabelModel, self).__init__()
        self.fc = torch.nn.Linear(num_classes, num_classes)

    def forward(self, x):
        return torch.sigmoid(self.fc(x))

# Initialize dataset and dataloader
dataset = MultiLabelDataset()
dataloader = DataLoader(dataset, batch_size=32, shuffle=True)

# Initialize model
model = MultiLabelModel()
criterion = torch.nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters())

# Training loop
for epoch in range(5):
    for data, labels in dataloader:
        optimizer.zero_grad()
        outputs = model(data)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

# Evaluation
model.eval()
y_true = []
y_pred = []

with torch.no_grad():
    for data, labels in dataloader:
        outputs = model(data)
        y_true.extend(labels.cpu().numpy())
        y_pred.extend((outputs.cpu().numpy() > 0.5).astype(int))

y_true = np.array(y_true)
y_pred = np.array(y_pred)

precision = precision_score(y_true, y_pred, average='micro')
recall = recall_score(y_true, y_pred, average='micro')
# Calculate the average precision score
average_precision = average_precision_score(y_true, y_pred, average="micro")

print("Average Precision Score: {:.4f}".format(average_precision))
print(f'Precision: {precision}')
print(f'Recall: {recall}')


In [ ]:
def compute_auc(labels, predictions):
    # Check inputs for errors.
    if len(predictions) != len(labels):
        raise Exception('Numbers of predictions and labels must be the same.')

    n = labels.shape[1]
    print(n)
    for i in range(n):
        if not labels[:, i] in (0, 1):
            raise Exception('Labels must satisfy label == 0 or label == 1.')

    for i in range(n):
        if not 0 <= predictions[:, i] <= 1:
            raise Exception('Predictions must satisfy 0 <= prediction <= 1.')

    # Find prediction thresholds.
    thresholds = np.unique(predictions)[::-1]
    if thresholds[0] != 1:
        thresholds = np.concatenate((np.array([1]), thresholds))

    if thresholds[-1] != 0:
        thresholds = np.concatenate((thresholds, np.array([0])))
    m = len(thresholds)

    # Populate contingency table across prediction thresholds.
    tp = np.zeros(m)
    fp = np.zeros(m)
    fn = np.zeros(m)
    tn = np.zeros(m)

    # Find indices that sort predicted probabilities from largest to smallest.
    idx = np.argsort(predictions)[::-1]

    i = 0
    for j in range(m):
        # Initialize contingency table for j-th prediction threshold.
        if j == 0:
            tp[j] = 0
            fp[j] = 0
            fn[j] = np.sum(labels)
            tn[j] = n - fn[j]
        else:
            tp[j] = tp[j - 1]
            fp[j] = fp[j - 1]
            fn[j] = fn[j - 1]
            tn[j] = tn[j - 1]

        # Update contingency table for i-th largest prediction probability.
        while i < n and predictions[idx[i]] >= thresholds[j]:
            if labels[idx[i]]:
                tp[j] += 1
                fn[j] -= 1
            else:
                fp[j] += 1
                tn[j] -= 1
            i += 1

    # Summarize contingency table.
    tpr = np.zeros(m)
    tnr = np.zeros(m)
    ppv = np.zeros(m)
    npv = np.zeros(m)

    for j in range(m):
        if tp[j] + fn[j]:
            tpr[j] = tp[j] / (tp[j] + fn[j])
        else:
            tpr[j] = 1
        if fp[j] + tn[j]:
            tnr[j] = tn[j] / (fp[j] + tn[j])
        else:
            tnr[j] = 1
        if tp[j] + fp[j]:
            ppv[j] = tp[j] / (tp[j] + fp[j])
        else:
            ppv[j] = 1
        if fn[j] + tn[j]:
            npv[j] = tn[j] / (fn[j] + tn[j])
        else:
            npv[j] = 1

    # Compute AUROC as the area under a piecewise linear function of TPR /
    # sensitivity (x-axis) and TNR / specificity (y-axis) and AUPRC as the area
    # under a piecewise constant of TPR / recall (x-axis) and PPV / precision
    # (y-axis).
    auroc = 0
    auprc = 0
    for j in range(m-1):
        auroc += 0.5 * (tpr[j + 1] - tpr[j]) * (tnr[j + 1] + tnr[j])
        auprc += (tpr[j + 1] - tpr[j]) * ppv[j + 1]

    return auroc, auprc


In [ ]:
thresholds = np.unique(y_pred)[::-1]

In [ ]:
thresholds = np.unique(y_pred)[::-1]
thresholds

In [ ]:
if thresholds[0] != 1:
    thresholds = np.concatenate((np.array([1]), thresholds))

    if thresholds[-1] != 0:
        thresholds = np.concatenate((thresholds, np.array([0])))
    m = len(thresholds)

    # Populate contingency table across prediction thresholds.
    tp = np.zeros(m)
    fp = np.zeros(m)
    fn = np.zeros(m)
    tn = np.zeros(m)

In [ ]:
auroc, auprc = compute_auc(y_true, y_pred)

In [ ]:
for i in range(y_true.shape[-1]):
    print(y_true[:, i])

In [ ]:
y_true.shape

In [ ]:
y_true.shape[-1]

In [ ]:
len(y_true)

In [ ]:
auroc, auprc

In [ ]:
y_true[0]

In [ ]:
y_pred[0]

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import heapq

def initialize_grid(grid_size):
    # Initialize a grid with random values
    grid = np.random.rand(grid_size, grid_size)
    invaded = np.zeros_like(grid, dtype=bool)
    return grid, invaded

def invasion_percolation(grid_size):
    grid, invaded = initialize_grid(grid_size)
    priority_queue = []
    distances = np.full_like(grid, np.inf)
    
    # Start the invasion from the top row
    for y in range(grid_size):
        heapq.heappush(priority_queue, (grid[0, y], 0, y, 0))
        invaded[0, y] = True
        distances[0, y] = 0
    
    while priority_queue:
        threshold, x, y, dist = heapq.heappop(priority_queue)
        
        # If the bottom row is reached, stop the process
        if x == grid_size - 1:
            break
        
        # Check the neighbors of the current cell
        for dx, dy in [(-1, 0), (1, 0), (0, -1), (0, 1)]:
            nx, ny = x + dx, y + dy
            if 0 <= nx < grid_size and 0 <= ny < grid_size and not invaded[nx, ny]:
                new_dist = dist + 1  # Increment the distance by 1 for each step
                invaded[nx, ny] = True
                distances[nx, ny] = min(distances[nx, ny], new_dist)
                heapq.heappush(priority_queue, (grid[nx, ny], nx, ny, new_dist))
    
    return grid, invaded, distances

def plot_invasion(grid, invaded, distances):
    plt.figure(figsize=(12, 10))
    cbar_kws = {"shrink":1,
           'extendfrac':.2, 
           "drawedges":False} 
    # Plot grid values
    plt.subplot(1, 2, 1)
    plt.title("Grid Values")
    plt.imshow(grid, cmap='viridis')
    #plt.colorbar(label='Threshold Value')
    
    # Plot invasion process
    plt.subplot(1, 2, 2)
    plt.title("Invasion Process and Distances")
    plt.imshow(invaded, cmap='Reds', alpha=0.5)
    #plt.colorbar(label='Invaded Cells')
    
    # Overlay distances
    for (i, j), val in np.ndenumerate(distances):
        if invaded[i, j]:
            plt.text(j, i, f'{val:.0f}', ha='center', va='center', color='black')
    plt.savefig("/home/sangaria/Music/codes/RectangularUnits.pdf", bbox_inches='tight', dpi=100)
    plt.show()

if __name__ == "__main__":
    grid_size = 100  # Size of the grid
    
    grid, invaded, distances = invasion_percolation(grid_size)
    plot_invasion(grid, invaded, distances)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import heapq

def initialize_grid(grid_size):
    # Initialize a grid with random values
    grid = np.random.rand(grid_size, grid_size)
    invaded = np.zeros_like(grid, dtype=bool)
    return grid, invaded

def invasion_percolation(grid_size):
    grid, invaded = initialize_grid(grid_size)
    priority_queue = []
    distances = np.full_like(grid, np.inf)
    
    # Start the invasion from the top row
    for y in range(grid_size):
        heapq.heappush(priority_queue, (grid[0, y], 0, y, 0))
        invaded[0, y] = True
        distances[0, y] = 0
    
    while priority_queue:
        threshold, x, y, dist = heapq.heappop(priority_queue)
        
        # If the bottom row is reached, stop the process
        if x == grid_size - 1:
            break
        
        # Check the neighbors of the current cell
        for dx, dy in [(-1, 0), (1, 0), (0, -1), (0, 1)]:
            nx, ny = x + dx, y + dy
            if 0 <= nx < grid_size and 0 <= ny < grid_size and not invaded[nx, ny]:
                new_dist = dist + 1  # Increment the distance by 1 for each step
                invaded[nx, ny] = True
                distances[nx, ny] = min(distances[nx, ny], new_dist)
                heapq.heappush(priority_queue, (grid[nx, ny], nx, ny, new_dist))
    
    return grid, invaded, distances

def plot_invasion(grid, invaded, distances):
    grid_size = grid.shape[0]
    x, y = np.meshgrid(np.arange(grid_size), np.arange(grid_size))
    
    fig, ax = plt.subplots(figsize=(12, 12))
    
    # Plot grid values with circular markers
    scatter = ax.scatter(y, x, c=grid, cmap='viridis', marker='o', s=500 / grid_size**0.5)
    #plt.colorbar(scatter, ax=ax, label='Threshold Value')
    
    # Overlay invasion process
    invaded_x, invaded_y = np.where(invaded)
    scatter_invaded = ax.scatter(invaded_y, invaded_x, facecolors='none', edgecolors='r', s=500 / grid_size**0.5)
    
    # Overlay distances
    for (i, j), val in np.ndenumerate(distances):
        if invaded[i, j]:
            ax.text(j, i, f'{val:.0f}', ha='center', va='center', color='black', fontsize=8)
    
    ax.invert_yaxis()
    ax.set_aspect('equal')
    plt.title('Invasion Percolation with Circular Markers')
     #plt.savefig("/home/sangaria/Music/codes/CircularUnitsMarkers.pdf", bbox_inches='tight', dpi=100)
    plt.show()

if __name__ == "__main__":
    grid_size = 50  # Size of the grid
    
    grid, invaded, distances = invasion_percolation(grid_size)
    plot_invasion(grid, invaded, distances)


In [ ]:
grid_v = np.random.rand(50, 50)
grid_v

In [ ]:
invaded_v = np.zeros_like(grid_v, dtype=bool)
invaded_v

In [ ]:
distances_v = np.full_like(grid_v, np.inf)
distances_v

In [ ]:
for y in range(50):
    print((grid_v[0, y], 0, y, 0))
    #heapq.heappush(priority_queue, (grid[0, y], 0, y, 0))

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import heapq

def initialize_grid(grid_size):
    # Initialize a grid with random values
    grid = np.random.rand(grid_size, grid_size)
    invaded = np.zeros_like(grid, dtype=bool)
    return grid, invaded

def invasion_percolation(grid_size):
    grid, invaded = initialize_grid(grid_size)
    priority_queue = []
    distances = np.full_like(grid, np.inf)
    
    # Start the invasion from the top row
    for y in range(grid_size):
        heapq.heappush(priority_queue, (grid[0, y], 0, y, 0))
        invaded[0, y] = True
        distances[0, y] = 0
    
    while priority_queue:
        threshold, x, y, dist = heapq.heappop(priority_queue)
        
        # If the bottom row is reached, stop the process
        if x == grid_size - 1:
            break
        
        # Check the neighbors of the current cell
        for dx, dy in [(-1, 0), (1, 0), (0, -1), (0, 1)]:
            nx, ny = x + dx, y + dy
            if 0 <= nx < grid_size and 0 <= ny < grid_size and not invaded[nx, ny]:
                new_dist = dist + 1  # Increment the distance by 1 for each step
                invaded[nx, ny] = True
                distances[nx, ny] = min(distances[nx, ny], new_dist)
                heapq.heappush(priority_queue, (grid[nx, ny], nx, ny, new_dist))
    
    return grid, invaded, distances

def plot_invasion(grid, invaded, distances):
    grid_size = grid.shape[0]
    x, y = np.meshgrid(np.arange(grid_size), np.arange(grid_size))
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(24, 12))
    
    # Plot grid values with circular markers
    scatter1 = ax1.scatter(y, x, c=grid, cmap='viridis', marker='o', s=500 / grid_size**0.5)
    plt.colorbar(scatter1, ax=ax1, label='Threshold Value')
    ax1.set_title('Grid Values with Circular Markers')
    ax1.invert_yaxis()
    ax1.set_aspect('equal')

    # Plot invasion process and distances
    scatter2 = ax2.scatter(y, x, c='lightgrey', marker='o', s=500 / grid_size**0.5, edgecolor='none')
    
    # Overlay invaded cells
    invaded_x, invaded_y = np.where(invaded)
    scatter_invaded = ax2.scatter(invaded_y, invaded_x, facecolors='none', edgecolors='r', s=500 / grid_size**0.5)
    
    # Overlay distances
    for (i, j), val in np.ndenumerate(distances):
        if invaded[i, j]:
            ax2.text(j, i, f'{val:.0f}', ha='center', va='center', color='black', fontsize=8)
    
    ax2.set_title('Invasion Process and Distances')
    ax2.invert_yaxis()
    ax2.set_aspect('equal')
    
    #plt.savefig("/home/sangaria/Music/codes/CircularUnitsMarkers.pdf", bbox_inches='tight')
    plt.show()

if __name__ == "__main__":
    grid_size = 50  # Size of the grid
    
    grid, invaded, distances = invasion_percolation(grid_size)
    plot_invasion(grid, invaded, distances)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import heapq

def invasion_percolation(grid_size, start_point):
    # Initialize the grid with random threshold values
    grid = np.random.rand(grid_size, grid_size)
    # Initialize a mask to track invaded sites
    invaded = np.zeros_like(grid, dtype=bool)
    # Use a priority queue to keep track of the invasion front
    priority_queue = []
    
    # Start the invasion from the start_point
    start_x, start_y = start_point
    heapq.heappush(priority_queue, (grid[start_x, start_y], start_x, start_y))
    invaded[start_x, start_y] = True
    
    while priority_queue:
        # Get the cell with the lowest threshold value
        threshold, x, y = heapq.heappop(priority_queue)
        
        # Check the neighbors of the current cell
        for dx, dy in [(-1, 0), (1, 0), (0, -1), (0, 1)]:
            nx, ny = x + dx, y + dy
            if 0 <= nx < grid_size and 0 <= ny < grid_size and not invaded[nx, ny]:
                invaded[nx, ny] = True
                heapq.heappush(priority_queue, (grid[nx, ny], nx, ny))
    
    return grid, invaded

def plot_invasion(grid, invaded):
    plt.figure(figsize=(10, 10))
    plt.imshow(grid, cmap='viridis', alpha=0.5)
    plt.imshow(invaded, cmap='Reds', alpha=0.5)
    plt.colorbar(label='Threshold Value')
    plt.title('Invasion Percolation')
    plt.show()

if __name__ == "__main__":
    grid_size = 300  # Size of the grid
    start_point = (grid_size // 2, grid_size // 2)  # Start from the center of the grid
    
    grid, invaded = invasion_percolation(grid_size, start_point)
    plot_invasion(grid, invaded)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import heapq

def is_within_circle(x, y, center, radius):
    return (x - center[0]) ** 2 + (y - center[1]) ** 2 <= radius ** 2

def initialize_grid(grid_size, radius):
    grid = np.random.rand(grid_size, grid_size)
    invaded = np.zeros_like(grid, dtype=bool)
    center = (grid_size // 2, grid_size // 2)
    
    # Mask out cells outside the circle
    for x in range(grid_size):
        for y in range(grid_size):
            if not is_within_circle(x, y, center, radius):
                grid[x, y] = np.inf  # Mark cells outside the circle as inaccessible
    
    return grid, invaded, center

def invasion_percolation(grid_size, radius):
    grid, invaded, center = initialize_grid(grid_size, radius)
    priority_queue = []
    
    # Start the invasion from the center of the grid
    start_x, start_y = center
    heapq.heappush(priority_queue, (grid[start_x, start_y], start_x, start_y))
    invaded[start_x, start_y] = True
    
    while priority_queue:
        threshold, x, y = heapq.heappop(priority_queue)
        
        # Check the neighbors of the current cell
        for dx, dy in [(-1, 0), (1, 0), (0, -1), (0, 1)]:
            nx, ny = x + dx, y + dy
            if 0 <= nx < grid_size and 0 <= ny < grid_size and not invaded[nx, ny] and grid[nx, ny] != np.inf:
                invaded[nx, ny] = True
                heapq.heappush(priority_queue, (grid[nx, ny], nx, ny))
    
    return grid, invaded

def plot_invasion(grid, invaded, radius):
    plt.figure(figsize=(10, 10))
    plt.imshow(grid, cmap='viridis', alpha=0.5)
    plt.imshow(invaded, cmap='Reds', alpha=0.5)
    
    # Draw the circular boundary
    circle = plt.Circle((grid.shape[0] // 2, grid.shape[1] // 2), radius, color='blue', fill=False, linewidth=2)
    plt.gca().add_artist(circle)
    
    plt.colorbar(label='Threshold Value')
    plt.title('Invasion Percolation with Circular Boundary')
    plt.show()

if __name__ == "__main__":
    grid_size = 100  # Size of the grid
    radius = 40  # Radius of the circle
    
    grid, invaded = invasion_percolation(grid_size, radius)
    plot_invasion(grid, invaded, radius)


In [ ]:
def data_token(token_inputs):
    print(len(token_inputs))
    x_train_in_ids = torch.tensor([token_inputs[i][4] for i in range(len(token_inputs))], dtype=torch.long)
    x_train_in_mask = torch.tensor([token_inputs[i][5] for i in range(len(token_inputs))], dtype=torch.long)
    x_train_in_token = torch.tensor([token_inputs[i][6] for i in range(len(token_inputs))], dtype=torch.long)
    
    x_train_in_ids_1 = torch.tensor([token_inputs[i][7] for i in range(len(token_inputs))], dtype=torch.long)
    x_train_in_mask_1 = torch.tensor([token_inputs[i][8] for i in range(len(token_inputs))], dtype=torch.long)
    x_train_in_token_1 = torch.tensor([token_inputs[i][9] for i in range(len(token_inputs))], dtype=torch.long)
    
    notes_train = {"input_ids": x_train_in_ids, "attention_mask": x_train_in_mask,
                   "token_type_ids": x_train_in_token}
    notes_train_1 = {"input_ids": x_train_in_ids_1, "attention_mask": x_train_in_mask_1,
                   "token_type_ids": x_train_in_token_1}
    return notes_train, notes_train_1

def dataloaders(train_x, train_t, train_last, train_freq, train_notes, train_notes_1, train_y,
                test_x, test_t, test_last, test_freq, test_notes, test_notes_1, test_y, BATCH_SIZE=64):
    def collate_fn(batch):
        return tuple(zip(*batch))
    
    train_dataset = TensorDataset(
        torch.tensor(train_x, dtype=torch.float32),
        torch.tensor(train_t, dtype=torch.float32),
        torch.tensor(train_last, dtype=torch.float32),
        torch.tensor(train_freq, dtype=torch.float32),
        train_notes['input_ids'],
        train_notes['attention_mask'],
        train_notes['token_type_ids'],
        train_notes_1['input_ids'],
        train_notes_1['attention_mask'], 
        train_notes_1['token_type_ids'],
        torch.tensor(train_y, dtype=torch.float32)
    )

    valid_dataset = TensorDataset(
        torch.tensor(test_x, dtype=torch.float32),
        torch.tensor(test_t, dtype=torch.float32),
        torch.tensor(test_last, dtype=torch.float32),
        torch.tensor(test_freq, dtype=torch.float32),
        test_notes['input_ids'], 
        test_notes['attention_mask'], 
        test_notes['token_type_ids'],
        test_notes_1['input_ids'], 
        test_notes_1['attention_mask'],
        test_notes_1['token_type_ids'], 
        torch.tensor(test_y, dtype=torch.float32)
    )

    train_data_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=4,
        collate_fn=collate_fn
    )

    valid_data_loader = DataLoader(
        valid_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=4,
        collate_fn=collate_fn
    )

    return train_data_loader, valid_data_loader
def cv_fold_splits(data, data_real, notes, notes1, time_data, last_data, features_freqs, target, n_fold=2):
    from sklearn.model_selection import StratifiedKFold
    from sklearn.model_selection import train_test_split
    test_data_loader, training_data, validation_data = [], [], []
    all_test_notes, all_train_notes, all_valid_notes = [], [], []
    kfold = StratifiedKFold(n_splits=n_fold, shuffle=True, random_state=n_fold)
    
    for index, (train_index, test_index) in enumerate(kfold.split(data, target)):
        print('<-------OUTTER FOLD', index + 1, '------->')
        x_train, x_test = data[train_index], data[test_index]
        _, x_test_real = data_real[train_index], data_real[test_index]
        x_train_not, x_test_not = {key: val[train_index] for key, val in notes.items()}, {key: val[test_index] for key, val in notes.items()}
        x_train_not_1, x_test_not_1 = {key: val[train_index] for key, val in notes1.items()}, {key: val[test_index] for key, val in notes1.items()}
        x_train_last, x_test_last = last_data[train_index], last_data[test_index]
        x_train_freq, x_test_freq = features_freqs[train_index], features_freqs[test_index]
        x_train_t, x_test_t = time_data[train_index], time_data[test_index]
        y_train, y_test = target[train_index], target[test_index]
        
        _, test_loader = dataloaders(
            x_train, x_train_t, x_train_last, x_train_freq, x_train_not, x_train_not_1, y_train, 
            x_test, x_test_t, x_test_last, x_test_freq, x_test_not, x_test_not_1, y_test
        )
        
        x_train_not_ids, x_train_not_ids_1 = x_train_not['input_ids'],  x_train_not_1['input_ids']
        x_train_not_mask, x_train_not_mask_1 = x_train_not['attention_mask'],  x_train_not_1['attention_mask']
        x_train_not_token, x_train_not_token_1 = x_train_not['token_type_ids'],  x_train_not_1['token_type_ids']

        X_train, X_valid, y_train_fold, y_valid_fold = train_test_split(
            list(zip(x_train, x_train_t, x_train_last, x_train_freq, x_train_not_ids, 
                     x_train_not_mask, x_train_not_token, x_train_not_ids_1, 
                     x_train_not_mask_1, x_train_not_token_1)),
            y_train, stratify=y_train, test_size=0.2, random_state=n_fold
        )
        
        # Training data
        x_train_in = np.array([X_train[i][0] for i in range(len(X_train))])
        x_train_in_t = np.array([X_train[i][1] for i in range(len(X_train))])
        x_train_in_last = np.array([X_train[i][2] for i in range(len(X_train))])
        x_train_in_freq = np.array([X_train[i][3] for i in range(len(X_train))])
        # Clinical notes
        notes_train_in, notes_train_1_in = data_token(X_train)
        
        # Validation data
        x_val_in = np.array([X_valid[i][0] for i in range(len(X_valid))])
        x_val_in_t = np.array([X_valid[i][1] for i in range(len(X_valid))])
        x_val_in_last = np.array([X_valid[i][2] for i in range(len(X_valid))])
        x_val_in_freq = np.array([X_valid[i][3] for i in range(len(X_valid))])
        # Clinical notes
        notes_val_in, notes_val_1_in = data_token(X_valid)
        
        train_loader, val_loader = dataloaders(
            x_train_in, x_train_in_t, x_train_in_last, x_train_in_freq, notes_train_in, notes_train_1_in, y_train_fold,
            x_val_in, x_val_in_t, x_val_in_last, x_val_in_freq, notes_val_in, notes_val_1_in, y_valid_fold
        )

        training_data.append(train_loader)
        validation_data.append(val_loader)
        all_test_notes.append(x_test_not)
        test_data_loader.append(test_loader)
    
    return test_data_loader, training_data, validation_data, all_test_notes, x_train_in.shape
